In [12]:
import os
os.environ["WANDB_DISABLED"] = "true"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [13]:
import sys
# !{sys.executable} -m pip install datasets transformers snac soundfile torchaudio
# !{sys.executable} -m pip install "accelerate>=1.1.0"
# !{sys.executable} -m pip install flash-attn --no-build-isolation

In [14]:
import torch
import yaml
from datasets import load_from_disk
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

config_file = "config.yaml"
with open(config_file, "r") as f:
    config = yaml.safe_load(f)

model_name = config["model_name"]
base_repo_id  = config["save_folder"]
epochs = config["epochs"]
batch_size = config["batch_size"]
save_steps = config["save_steps"]
learning_rate = config["learning_rate"]

TOKENISED_DATASET = "./colombian_tokenised"

print(f"./{base_repo_id}")

./checkpoints


In [15]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

special_tokens = {
    "additional_special_tokens": [
        "<custom_token_1>",
        "<custom_token_2>",
        "<custom_token_3>",
        "<custom_token_4>",
        "<custom_token_5>",
        "<custom_token_6>",
        "<custom_token_7>",
    ]
}

tokenizer.add_special_tokens(special_tokens)
print(f"Vocabulario con tokens especiales: {len(tokenizer)}")

Vocabulario con tokens especiales: 156940


In [16]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    attn_implementation="flash_attention_2",
)

model.resize_token_embeddings(len(tokenizer))
print(f"Modelo: {next(model.parameters()).device}")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/255 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`
The new lm_head weights will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Modelo: cuda:0


In [17]:
ds = load_from_disk(TOKENISED_DATASET)
ds.set_format(type="torch")

print(f"Total ejemplos: {len(ds)}")
print(f"Columnas: {ds.column_names}")
print(f"Longitud promedio input_ids: {sum(len(x) for x in ds['input_ids']) / len(ds):.0f} tokens")

Total ejemplos: 2369
Columnas: ['input_ids', 'labels', 'attention_mask']
Longitud promedio input_ids: 446 tokens


In [18]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    pad_to_multiple_of=8,
)

In [19]:
training_args = TrainingArguments(
    num_train_epochs=15,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    logging_steps=1,
    bf16=True,
    output_dir=f"./{base_repo_id}",
    report_to="none",
    remove_unused_columns=False, 
    learning_rate=1e-4,
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    save_steps=500,
    save_total_limit=2,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [20]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=ds,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
/usr/local/lib/python3.11/dist-packages/transformers/data/data_collator.py:600: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  batch["labels"] = torch.tensor(batch["labels"], dtype=torch.int64)
/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Step,Training Loss
1,5.208358
2,5.178543
3,5.159806
4,5.136963
5,5.255999
6,5.208400
7,5.242248
8,5.088113
9,5.126806
10,5.039128


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/torch/utils/checkpoint.py:1399: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with device_autocast_ctx, torch.cpu.amp.autocast(**cpu_autocast_kwargs), recompute_context:  # type: ignore[attr-defined]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2235, training_loss=1.28427673984794, metrics={'train_runtime': 5885.348, 'train_samples_per_second': 6.038, 'train_steps_per_second': 0.38, 'total_flos': 3.131585958551224e+17, 'train_loss': 1.28427673984794, 'epoch': 15.0})

In [21]:

FINAL_DIR = f"./{base_repo_id}/final"

model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

FINAL_DIR

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

'./checkpoints/final'

## Inferencia — Generar audio

In [22]:
import torch
import soundfile as sf
import numpy as np
from snac import SNAC
from transformers import AutoModelForCausalLM, AutoTokenizer

CHECKPOINT = f"./{base_repo_id}/final"

tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/snac/snac.py:108: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(model_path, map_location="cpu")


In [23]:
import torch
import soundfile as sf
from snac import SNAC
from transformers import AutoModelForCausalLM, AutoTokenizer

tokeniser_length = 128256
start_of_text = 128000
end_of_text = 128009
start_of_speech = tokeniser_length + 1
end_of_speech = tokeniser_length + 2
start_of_human = tokeniser_length + 3
end_of_human = tokeniser_length + 4
start_of_ai = tokeniser_length + 5
end_of_ai = tokeniser_length + 6

CHECKPOINT = f"./{base_repo_id}/final"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

model.eval()
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cuda")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

In [41]:
tokeniser_length = 128256
start_of_human = tokeniser_length + 3
end_of_human = tokeniser_length + 4
start_of_ai = tokeniser_length + 5
start_of_speech = tokeniser_length + 1
end_of_speech = tokeniser_length + 2
end_of_text = 128009

def generate_audio(text, max_new_tokens=1200, temperature=0.1):
    tagged_text = f"colombia: {text}"
    text_ids = tokenizer.encode(tagged_text, add_special_tokens=True)
    text_ids.append(end_of_text)

    input_ids = torch.tensor(
        [[start_of_human] + text_ids + [end_of_human] + [start_of_ai] + [start_of_speech]]
    ).to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=[end_of_speech, end_of_ai],
        )

    generated = output_ids[0][input_ids.shape[1]:].tolist()
    cut = len(generated)
    for i, t in enumerate(generated):
        if t in [end_of_speech, end_of_ai]:
            cut = i
            break

    audio_tokens = generated[:cut]
    return audio_tokens


def redistribute_codes(code_list, snac_model):
    layer_1 = []
    layer_2 = []
    layer_3 = []
    for i in range(len(code_list) // 7):
        layer_1.append(code_list[7 * i])
        layer_2.append(code_list[7 * i + 1] - 4096)
        layer_3.append(code_list[7 * i + 2] - (2 * 4096))
        layer_3.append(code_list[7 * i + 3] - (3 * 4096))
        layer_2.append(code_list[7 * i + 4] - (4 * 4096))
        layer_3.append(code_list[7 * i + 5] - (5 * 4096))
        layer_3.append(code_list[7 * i + 6] - (6 * 4096))

    for name, layer in [("c0", layer_1), ("c1", layer_2), ("c2", layer_3)]:
        out_of_range = [v for v in layer if v < 0 or v >= 4096]
        if out_of_range:
            raise ValueError(f"{name} tiene {len(out_of_range)} tokens fuera de rango: {out_of_range[:5]}")

    codes = [
        torch.tensor(layer_1).unsqueeze(0).to("cuda"),
        torch.tensor(layer_2).unsqueeze(0).to("cuda"),
        torch.tensor(layer_3).unsqueeze(0).to("cuda"),
    ]

    with torch.no_grad():
        audio_hat = snac_model.decode(codes)
    return audio_hat


def tokens_to_audio(audio_tokens):
    end_of_speech = 128258
    audio_tokens = [t for t in audio_tokens if t != end_of_speech]
    new_length = (len(audio_tokens) // 7) * 7
    audio_tokens = audio_tokens[:new_length]
    
    code_list = [t - 128266 for t in audio_tokens]
    audio = redistribute_codes(code_list, snac_model)

    return audio.cpu().numpy().squeeze()


def generate_audio_safe(text, max_attempts=3, **kwargs):
    for attempt in range(max_attempts):
        try:
            audio_tokens = generate_audio(text, **kwargs)
            waveform = tokens_to_audio(audio_tokens)
            return waveform

        except (ValueError, RuntimeError) as e:
            print(f"Intento {attempt+1} fallido: {e}")
            torch.cuda.empty_cache()
            if attempt == max_attempts - 1:
                raise

    return None

In [53]:
import IPython.display as ipd
import soundfile as sf

text = "Le explico que incluye y seguro cambia de opinión. No es solo el celular, sino todo el apoyo y garantía que manejámos."
waveform = generate_audio_safe(text, max_attempts=3)

sf.write("output_colombiano.wav", waveform, 24000)
ipd.display(ipd.Audio(waveform, rate=24000, autoplay=False))

In [43]:
import IPython.display as ipd
import soundfile as sf

text = "Estoy estudiando inteligencia artificial en KeepCoding. Qué cosa mas buena."
waveform = generate_audio_safe(text, max_attempts=3)

sf.write("output_colombiano2.wav", waveform, 24000)
ipd.display(ipd.Audio(waveform, rate=24000, autoplay=False))

In [52]:
import IPython.display as ipd
import soundfile as sf

text = "Buen dia Bienvenido a KeeShop, ¿en qué le puedo ayudar hoy?."
waveform = generate_audio_safe(text, max_attempts=3)

sf.write("output_colombiano3.wav", waveform, 24000)
ipd.display(ipd.Audio(waveform, rate=24000, autoplay=False))

Intento 1 fallido: c0 tiene 11 tokens fuera de rango: [22863, 24877, 24962, 28227, 26387]
